### Invoice Multi-Agent Supervisor

Creates an Agent Bricks Supervisor that coordinates three specialized agents:
- **Genie space** (`procurement-analytics`) — structured SQL queries over silver and gold Delta tables
- **Invoice Knowledge Assistant** (`invoice-document-search`) — PDF Q&A over 14 supplier invoice PDFs
- **Contract Knowledge Assistant** (`contract-document-search`) — PDF Q&A over 10 vendor contract PDFs

Routes queries based on intent: numbers and aggregates go to Genie, invoice-specific
detail questions go to the Invoice KA, and contract terms and compliance questions
go to the Contract KA.

In [ ]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
INVOICE_KNOWLEDGE_ENDPOINT = dbutils.widgets.get("INVOICE_KNOWLEDGE_ENDPOINT_NAME")
CONTRACT_KNOWLEDGE_ENDPOINT = dbutils.widgets.get("CONTRACT_KNOWLEDGE_ENDPOINT_NAME")
SUPERVISOR_ENDPOINT = dbutils.widgets.get("INVOICE_SUPERVISOR_ENDPOINT_NAME")

##### Retrieve the Genie space ID from uc_state

In [ ]:
import json

state_df = spark.sql(f"""
    SELECT resource_data
    FROM {CATALOG}._internal_state.resources
    WHERE resource_type = 'genie_spaces'
    ORDER BY created_at DESC
    LIMIT 1
""")

if state_df.count() > 0:
    genie_info = json.loads(state_df.first().resource_data)
    GENIE_SPACE_ID = genie_info.get("space_id", "")
    print(f"Found Genie space: {GENIE_SPACE_ID}")
else:
    raise RuntimeError("No Genie space found in uc_state. Ensure invoice_genie stage ran first.")

##### Create the Supervisor Agent

In [ ]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

def get_tile_by_name(name):
    resp = w.api_client.do("GET", "/api/2.0/tiles")
    for tile in resp.get("tiles", []):
        if tile.get("name") == name:
            return tile
    return None

def get_endpoint_id_by_name(agent_name):
    tile = get_tile_by_name(agent_name)
    if tile:
        return tile["serving_endpoint_name"]
    raise RuntimeError(
        f"No tile found with name '{agent_name}'. "
        f"Ensure the knowledge agent stage completed successfully."
    )

INVOICE_KA_NAME = f"{CATALOG}-invoice-knowledge"
CONTRACT_KA_NAME = f"{CATALOG}-contract-knowledge"

INVOICE_ENDPOINT_ID = get_endpoint_id_by_name(INVOICE_KA_NAME)
print(f"Resolved invoice knowledge endpoint: {INVOICE_KA_NAME} -> {INVOICE_ENDPOINT_ID}")

CONTRACT_ENDPOINT_ID = get_endpoint_id_by_name(CONTRACT_KA_NAME)
print(f"Resolved contract knowledge endpoint: {CONTRACT_KA_NAME} -> {CONTRACT_ENDPOINT_ID}")

AGENT_NAME = f"{CATALOG}-invoice-supervisor"
API_BASE = "/api/2.0/multi-agent-supervisors"

SUPERVISOR_BODY = {
    "name": AGENT_NAME,
    "description": (
        "Caspers Kitchens procurement assistant that coordinates structured spend analytics, "
        "invoice document retrieval, and contract compliance review across 10 suppliers "
        "and 14 invoices. Routes analytics questions to the Genie space, invoice-specific "
        "detail questions to the Invoice Knowledge Assistant, and contract term or "
        "compliance questions to the Contract Knowledge Assistant."
    ),
    "endpoint_name": SUPERVISOR_ENDPOINT,
    "agents": [
        {
            "agent_type": "genie",
            "genie_space": {"id": GENIE_SPACE_ID},
            "name": "procurement-analytics",
            "description": (
                "Queries the structured procurement database for spend totals, invoice counts, "
                "payment aging, discount capture rates, supplier scorecards, and exception summaries. "
                "Use for questions that need numbers, aggregations, comparisons, or filtering "
                "across multiple invoices or suppliers — e.g. total overdue balance, spend by category, "
                "which supplier has the most discrepancies, discount capture rate."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": INVOICE_ENDPOINT_ID},
            "name": "invoice-document-search",
            "description": (
                "Searches the original supplier invoice PDF documents for specific invoice details: "
                "exact line items, unit prices billed, payment instructions, alert notices, "
                "discount or penalty calculations as printed on the invoice. "
                "Use for questions about a specific invoice number, what a supplier charged "
                "for a particular item, or what the invoice itself says about a discrepancy or penalty."
            ),
        },
        {
            "agent_type": "ka",
            "serving_endpoint": {"name": CONTRACT_ENDPOINT_ID},
            "name": "contract-document-search",
            "description": (
                "Searches vendor supply contract PDFs for the authoritative contract terms: "
                "contracted unit prices in Schedule A, payment terms (Net 30 / 2/10 Net 30 etc.), "
                "volume discount thresholds and rates, SLA delivery day commitments and penalty rates, "
                "late payment penalty clauses, price stability provisions, and dispute resolution steps. "
                "Use for questions about what a contract says, whether an invoice price matches "
                "the contract, what penalty applies for a given situation, or what the dispute process is."
            ),
        },
    ],
    "instructions": (
        "You are a procurement and accounts payable assistant for Caspers Kitchens. "
        "Always cite the invoice number, contract number, and supplier name when answering. "
        "Use Procurement Analytics for spend totals, rankings, and cross-supplier comparisons. "
        "Use Invoice Document Search for line-item details from a specific invoice. "
        "Use Contract Document Search for payment terms, contracted prices, SLA commitments, "
        "and penalty clauses. "
        "For questions that span both — e.g. 'Is this invoice price correct per the contract?' — "
        "query both the invoice and the contract and compare them explicitly."
    ),
}

def find_existing_id():
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type IN ('multi_agent_supervisors', 'endpoints')
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            info = json.loads(row.resource_data)
            if info.get("endpoint_name") == SUPERVISOR_ENDPOINT:
                return info.get("tile_id") or info.get("agent_id")
    except Exception:
        pass
    return None

def try_get(agent_ref):
    try:
        return w.api_client.do("GET", f"{API_BASE}/{agent_ref}")
    except Exception:
        return None

existing_id = find_existing_id()
agent_id = None
needs_polling = True

if existing_id:
    info = try_get(existing_id)
    if info:
        try:
            print(f"Supervisor exists ({existing_id}), updating...")
            w.api_client.do("PUT", f"{API_BASE}/{existing_id}", body=SUPERVISOR_BODY)
            agent_id = existing_id
            needs_polling = False
            print(f"\u2705 Updated Supervisor Agent: {agent_id}")
        except Exception as e:
            print(f"\u26a0\ufe0f Update failed ({type(e).__name__}: {e}), will recreate supervisor.")

if not agent_id:
    try:
        supervisor = w.api_client.do("POST", API_BASE, body=SUPERVISOR_BODY)
        print(f"POST response keys: {list(supervisor.keys())}")
        print(f"\u2705 Created Supervisor Agent")
    except Exception as e:
        err = str(e)
        if "already exists" in err.lower() or "ALREADY_EXISTS" in err:
            print(f"\u267b\ufe0f Supervisor {AGENT_NAME} already exists. Proceeding.")
        else:
            raise

tile = get_tile_by_name(AGENT_NAME)
if tile:
    agent_id = tile["tile_id"]
    print(f"Resolved tile_id: {agent_id}")
else:
    agent_id = AGENT_NAME
    needs_polling = False
    print(f"\u26a0\ufe0f Could not resolve tile_id, using name as fallback: {agent_id}")

In [ ]:
import time

if needs_polling:
    if not agent_id:
        print(f"\u26a0\ufe0f No agent_id available — skipping polling. Check POST response above.")
    else:
        MAX_WAIT = 300
        POLL_INTERVAL = 30
        elapsed = 0
        print(f"Checking if Supervisor endpoint is ready (agent_id={agent_id}, max {MAX_WAIT}s)...")

        while elapsed < MAX_WAIT:
            try:
                sup_status = w.api_client.do("GET", f"{API_BASE}/{agent_id}")
                inner = sup_status.get("multi_agent_supervisor", sup_status)
                ep_status = inner.get("status", {}).get("endpoint_status", "") if isinstance(inner.get("status"), dict) else inner.get("endpoint_status", "")
                print(f"  endpoint_status: {ep_status}")
                if str(ep_status).upper() in ("ACTIVE", "READY", "ONLINE"):
                    print(f"\u2705 Supervisor {AGENT_NAME} is READY")
                    break
            except Exception as e:
                print(f"  GET status check failed: {type(e).__name__}: {e}")

            time.sleep(POLL_INTERVAL)
            elapsed += POLL_INTERVAL
        else:
            print(f"\u23f3 Endpoint may still be provisioning — proceeding.")
else:
    print(f"\u2705 Supervisor already running — skipping polling.")

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

add(CATALOG, "multi_agent_supervisors", {"endpoint_name": SUPERVISOR_ENDPOINT, "tile_id": agent_id, "name": AGENT_NAME})
print("\u2705 Invoice Supervisor stage complete")